# 🧭 Data Preprocessing & External Data Integration
This notebook executes the preprocessing and external data loading steps

In [ ]:
# --- Imports ---
import pandas as pd
import glob
import os
from scripts import data_normalization, external

print('✅ Modules imported successfully.')

✅ Modules imported successfully.


In [ ]:
# Combine all raw ERP data
# Define input/output directories
FILE_DIR = "data/raw"
OUTPUT_PATH = "data/raw/combined"
OUTPUT_FILE = os.path.join(OUTPUT_PATH, "erp_order_data.xlsx")

# Ensure output folder exists
os.makedirs(FILE_DIR, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Define Finnish-to-English column mapping
COLUMN_MAP = {
    "Tilaus": "Order_Number",
    "As.tilausnro": "Customer_Order_Number",
    "Asiakas": "Customer_Number",
    "Asiakkaan nimi": "Customer_Name",
    "Nim.tunnus": "Product_Item_Number",
    "Tuoteryhmä": "Product_Item_Group",
    "Tuoteryh. nimi": "Group_Name",
    "Nim.nimi (pitkä)": "Product_Item_Name",
    "Til.pvm.": "Order_Date",
    "Toim.pvm": "Planned_Delivery_Date",
    "Ulk.toim.pvm": "Confirmed_Delivery_Date",
    "Tila": "Status",
    "Alkup.toim.pvm": "Original_Delivery_Date",
    "Edel.toim.pvm": "Last_Delivery_Date",
    "Määrä": "Ordered_Quantity",
    "Toim.": "Delivery_Quantity",
    "Jäljellä": "Quantity_Left",
    "Yks.": "Unit",
    "Hinta": "Unit_Price",
    "Arvo": "Order_Amount",    
    # --- Add these if available in your ERP exports ---
}

# Initialize a list to store dataframes
all_dataframes = []

# Pattern to match all yearly files in the raw folder
file_pattern = os.path.join(FILE_DIR, "SalesOrderRows_20*.xlsx")

# Loop through all Excel files
for file in glob.glob(file_pattern):
    print(f"Processing {file}...")
    
    # Extract year from filename, e.g., SalesOrderRows_2015.xlsx -> 2015
    year = os.path.splitext(os.path.basename(file))[0].split("_")[-1]

    # Read only the required columns
    df = pd.read_excel(file, usecols=list(COLUMN_MAP.keys()))
    
    # Rename columns to English
    df.rename(columns=COLUMN_MAP, inplace=True)
    
    # Add 'Order Year' column at the beginning
    df.insert(0, "Order_Year", int(year))
    
    # Append to list
    all_dataframes.append(df)

# Combine all DataFrames
merged_df = pd.concat(all_dataframes, ignore_index=True)

# Save merged data to cleaned folder
merged_df.to_excel(OUTPUT_FILE, sheet_name="all_orders", index=False)

print(f"\n✅ Successfully merged all ERP files into: {OUTPUT_FILE}")

Processing data/raw\SalesOrderRows_2015.xlsx...
Processing data/raw\SalesOrderRows_2016.xlsx...
Processing data/raw\SalesOrderRows_2017.xlsx...
Processing data/raw\SalesOrderRows_2018.xlsx...
Processing data/raw\SalesOrderRows_2019.xlsx...
Processing data/raw\SalesOrderRows_2020.xlsx...
Processing data/raw\SalesOrderRows_2021.xlsx...
Processing data/raw\SalesOrderRows_2022.xlsx...
Processing data/raw\SalesOrderRows_2023.xlsx...
Processing data/raw\SalesOrderRows_2024.xlsx...

✅ Successfully merged all ERP files into: data/raw/combined_new\erp_order_data.xlsx


In [ ]:
# --- Function: preprocess_data() ---
def normalize_data():
    """Preprocess ERP order data and export aggregated results."""
    df = pd.read_excel('data/raw/combined/erp_order_data.xlsx', 'all_orders')
    df = data_normalization.filter_products(df, 'data/raw/product_group.xlsx')
    df = data_normalization.handle_duplicate_value(df)
    df = data_normalization.standardize_data(df)

    data_normalization.check_missing_value(df)
    data_normalization.display_outliers(df)

    df.to_excel("data/inputs/erp_data_combined.xlsx", sheet_name="all_orders", index=False)
    print('💾 Saved Cleaned ERP data to data/inputs/erp_data_combined.xlsx')
    return df

In [ ]:
# --- Function: load_external_data() ---
def load_external_data():
    """Load and merge all external economic and industry datasets."""
    df_external = external.load_external_data()
    print('✅ External data loaded successfully.')
    print(df_external.head())
    # Uncomment below to save
    df_external.to_excel('data/inputs/external_data_combined.xlsx', sheet_name='ext_data', index=False)
    print('💾 Saved external data to data/input/external_data_combined.xlsx')
    return df_external

In [ ]:
# --- Run preprocessing ---
df_normalized = normalize_data()
df_normalized.head()

✅ Filtered ERP data to 17469 cylinder records (from 25199 total).
Number of duplicate rows: 101
Missing values per column:
 Order_Year                    0
Order_Number                  0
Customer_Order_Number      1709
Customer_Number               0
Customer_Name                 0
Product_Item_Number           0
Product_Item_Group            0
Product_Item_Name             0
Order_Date                    0
Planned_Delivery_Date         0
Confirmed_Delivery_Date      13
Status                        0
Original_Delivery_Date       71
Last_Delivery_Date           12
Ordered_Quantity              0
Delivery_Quantity            12
Quantity_Left                 0
Unit                          3
Order_Amount                  0
Group_Name                    0
Unit_Price                    0
dtype: int64
       Ordered_Quantity   Order_Amount    Unit_Price
count       17368.00000   17368.000000  17368.000000
mean           22.66398    3592.380833    260.153729
std            25.67516    5565.

,Order_Year,Order_Number,Customer_Order_Number,Customer_Number,Customer_Name,Product_Item_Number,Product_Item_Group,Product_Item_Name,Order_Date,Planned_Delivery_Date,...,Status,Original_Delivery_Date,Last_Delivery_Date,Ordered_Quantity,Delivery_Quantity,Quantity_Left,Unit,Order_Amount,Group_Name,Unit_Price
10,2015,CO500007,NaN,9999,MUUT ASIAKKAAT,5002,05002,Tappikiinnitteinen Sylinteri,2015-04-07,2015-04-08,...,laskutettu,2015-04-08,2015-04-07,1.0,1.0,0.0,kpl,282.26,Tappikiinn.Sylin.,282.26
11,2015,CO500008,NaN,9999,MUUT ASIAKKAAT,N02460585,05002,Sylinteri 60/40-870 6040870,2015-04-07,2015-04-07,...,laskutettu,2015-04-07,2015-04-07,1.0,1.0,0.0,kpl,240.00,Tappikiinn.Sylin.,240.00
39,2015,GR500003,NaN,21822,OUTOKUMMUN METALLI,V100001,05002,"0hh10509005502470479 90/55-247,5 a479",2015-02-11,2015-03-12,...,toimitettu,2015-03-12,2015-03-25,2.0,2.0,0.0,kpl,370.46,Tappikiinn.Sylin.,185.23
40,2015,GR500003,NaN,21822,OUTOKUMMUN METALLI,F673812,05002,0pp08007004001720386 70/40-172 a386,2015-02-11,2015-03-12,...,toimitettu,2015-03-12,2015-04-07,1.0,1.0,0.0,kpl,140.86,Tappikiinn.Sylin.,140.86
51,2015,GR500015,NaN,9999,MUUT ASIAKKAAT,08005006500005,05005,Sylinteri 80/50-650 A824,2015-05-22,2015-05-22,...,toimitettu,2015-05-22,2015-05-22,1.0,1.0,0.0,kpl,75.00,Erikoissylin.,75.00


In [ ]:
# --- Run external data loading ---
df_external = load_external_data()
df_external.head()

🔗 Merging All external data ...
✅ External data loaded successfully.
  Order_Month  Food_Price_Index  Electricity_Price  ECB_Inflation  \
0  2021-01-01            102.86           0.083143            0.9   
1  2021-02-01            103.18           0.086634            0.9   
2  2021-03-01            103.34           0.090124            1.3   
3  2021-04-01            103.48           0.093615            1.6   
4  2021-05-01            103.17           0.097106            2.0   

   ECB_Interest_Rate  FEDFUNDS_Value  Purchase_Index  Total_New_Orders_Value  \
0             -0.378            0.09             NaN            10002.500000   
1             -0.218            0.08             NaN            10203.100000   
2             -0.126            0.07             NaN            10403.700000   
3             -0.079            0.07             NaN            10604.300000   
4              0.047            0.06             NaN            10544.733333   

   Steel_Price  
0        213.1  
1

,Order_Month,Food_Price_Index,Electricity_Price,ECB_Inflation,ECB_Interest_Rate,FEDFUNDS_Value,Purchase_Index,Total_New_Orders_Value,Steel_Price
0,2021-01-01,102.86,0.083143,0.9,-0.378,0.09,NaN,10002.500000,213.1
1,2021-02-01,103.18,0.086634,0.9,-0.218,0.08,NaN,10203.100000,193.9
2,2021-03-01,103.34,0.090124,1.3,-0.126,0.07,NaN,10403.700000,210.5
3,2021-04-01,103.48,0.093615,1.6,-0.079,0.07,NaN,10604.300000,208.7
4,2021-05-01,103.17,0.097106,2.0,0.047,0.06,NaN,10544.733333,236.2


### 🎉 Pipeline completed successfully!
You can now proceed to notebook 01 for data cleaning.